## **설정**

In [ ]:
## Google Drive Amount
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
## Import libaries
import os

import pandas as pd
import numpy as np
import random

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler

In [ ]:
## Setting
base_path = '/content/drive/MyDrive/2025-2 통계계산특론/우리코드/'

## **데이터 불러오기**

In [ ]:
file_path = os.path.join(base_path, '00_HSData', 'CNN_macro')
years = [2018, 2019, 2020, 2021]

macro_sizes = [5, 10, 20, 50]
data_types = ['train', 'valid', 'test']

hidden_states_data = {}

for macro_size in macro_sizes:
    hidden_states_data[macro_size] = {}
    for data_type in data_types:
        hidden_states_data[macro_size][data_type] = {}
        for year in years:
            folder_name = f'Test_{year}'
            file_name = f'hidden_states_{data_type}_{macro_size}.csv'
            full_file_path = os.path.join(file_path, folder_name, file_name)
            hidden_states_data[macro_size][data_type][year] = pd.read_csv(full_file_path)

KeyboardInterrupt: 

In [ ]:
print(hidden_states_data[5]['train'][2018].head())

      sasdate    hs_5_0    hs_5_1    hs_5_2    hs_5_3    hs_5_4
0  1997-12-01  0.623487 -0.024417  0.456957 -0.004858  0.039064
1  1998-01-01  0.618013 -0.024752  0.451041 -0.000899  0.031257
2  1998-02-01  0.607670 -0.026278  0.439830 -0.004085  0.034021
3  1998-03-01  0.586814 -0.028085  0.435707 -0.006124  0.026002
4  1998-04-01  0.580117 -0.027980  0.421432 -0.007030  0.017517


In [ ]:
def set_seed(val):
    torch.manual_seed(val)
    torch.cuda.manual_seed(val)
    np.random.seed(val)
    random.seed(val)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

# 저장 경로 설정
SAVE_BASE_DIR = os.path.join(base_path, '00_HSData', 'CNN_BiLSTM_macro')


# 입력 크기별 출력 타겟 매핑
# Input Size(CNN Output) -> [Output Sizes(LSTM Output)]
size_mapping = {
    5: [5],
    10: [5, 10],
    20: [5, 10, 20],
    50: [5, 10, 20, 50]
}

In [ ]:
# --- 2. Multi-Head Bi-LSTM Encoder 모델 정의 ---
class MultiHeadBiLSTMEncoder(nn.Module):
    def __init__(self, input_dim, output_dims):
        super(MultiHeadBiLSTMEncoder, self).__init__()
        self.output_dims = output_dims

        # Shared Bi-LSTM
        # 1:1 매핑이므로 Sequence Length = 1로 들어옴
        # 이 경우 LSTM은 복잡한 Dense Layer처럼 동작함 (Context 정보 X)
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=input_dim,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )

        # Projections
        self.heads = nn.ModuleList()
        for out_dim in output_dims:
            # Bi-LSTM output (input_dim * 2) -> Target dim
            self.heads.append(nn.Linear(input_dim * 2, out_dim))

    def forward(self, x):
        # x shape: (Batch, 1, Input_Dim)

        # LSTM 통과
        # output shape: (Batch, 1, Hidden*2)
        lstm_out, _ = self.lstm(x)

        # 차원 축소: (Batch, Hidden*2)
        last_step_feature = lstm_out[:, -1, :]

        results = {}
        for i, head in enumerate(self.heads):
            # Projection
            out = head(last_step_feature)
            results[f'hs_{self.output_dims[i]}'] = out

        return results

In [ ]:
# macro_sizes 루프 (5, 10, 20, 50)
for macro_size in macro_sizes:
    target_dims = size_mapping[macro_size]
    print(f"\n[Processing macro Size: {macro_size} -> Targets: {target_dims}]")

    # 모델 초기화 (입력 차원 = macro_size)
    model = MultiHeadBiLSTMEncoder(input_dim=macro_size, output_dims=target_dims)
    model.eval()

    # Data Type 루프 (train, valid, test)
    for data_type in data_types:
        for year in years:
            # 1. 데이터 가져오기
            try:
                df = hidden_states_data[macro_size][data_type][year]
            except KeyError:
                continue

            if df.empty:
                continue

            # 2. Feature와 Meta 분리

            feature_cols_for_tensor = [c for c in df.columns if c.startswith('hs_')]
            meta_cols_for_df = [c for c in df.columns if not c.startswith('hs_')]

            # 혹시 모를 정렬
            if 'sasdate' in df.columns:
                df = df.sort_values('sasdate').reset_index(drop=True)

            # Feature 추출 - now guaranteed to be numerical
            feature_data = df[feature_cols_for_tensor].values

            # 3. 텐서 변환 (Windowing 없음)
            # LSTM 입력 규격 (Batch, Sequence, Input_Dim)을 맞추기 위해
            # Sequence Length를 1로 설정 -> (N, 1, Feature_Dim)
            X_tensor = torch.FloatTensor(feature_data).unsqueeze(1)

            # 메타 데이터는 그대로 사용 (행 개수 변화 없음)
            meta_data = df[meta_cols_for_df].copy()

            # 4. 모델 실행
            with torch.no_grad():
                outputs = model(X_tensor)

            # 5. 저장
            save_folder = os.path.join(SAVE_BASE_DIR, f'Test_{year}')
            os.makedirs(save_folder, exist_ok=True)

            for dim in target_dims:
                # 추출된 Hidden State
                feats = outputs[f'hs_{dim}'].numpy()
                feat_cols = [f'hs_{dim}_{k}' for k in range(dim)]

                # DataFrame 생성
                res_df = pd.DataFrame(feats, columns=feat_cols)

                # 메타 데이터 붙이기 (1:1 매핑이므로 바로 concat)
                res_df = pd.concat([meta_data, res_df], axis=1)

                # 파일명 설정
                file_name = f'hidden_states_{data_type}_{macro_size}_{dim}_macro.csv'
                save_path = os.path.join(save_folder, file_name)

                res_df.to_csv(save_path, index=False)

    print(f" -> macro Size {macro_size} 완료")

print("\nAll tasks completed.")


[Processing macro Size: 5 -> Targets: [5]]
 -> macro Size 5 완료

[Processing macro Size: 10 -> Targets: [5, 10]]
 -> macro Size 10 완료

[Processing macro Size: 20 -> Targets: [5, 10, 20]]
 -> macro Size 20 완료

[Processing macro Size: 50 -> Targets: [5, 10, 20, 50]]
 -> macro Size 50 완료

All tasks completed.
